In [1]:
import ast
import logging
import os.path
import re
from glob import glob

import yaml

# remove old log file
if os.path.exists('check_dataset_w_emg.log'):
    os.remove('check_dataset_w_emg.log')
logging.basicConfig(filename='check_dataset_w_emg.log', level=logging.INFO, format='%(message)s', force=True)

log_path = '../logs/dataset_w_emg'

psg_recording_regex = r'.*\d+/\d+ Processing (.*)'
hypno_recording_regex = r'.*\d+/\d+ Processing (.*)'
psg_channels_regex = r'.*Extracted (\d+) channels: (.*)'

dataset_paths = glob(os.path.join(log_path, '*'))
dataset_paths.sort()

for dataset_path in dataset_paths:
    dataset_name = dataset_path.split('/')[-1]
    logging.info(dataset_name)

    psg_log = glob(os.path.join(dataset_path, '*', 'extract_psg.log'))[-1]
    hypno_log = glob(os.path.join(dataset_path, '*', 'extract_hypno.log'))[-1]

    # check if all recordings have both PSG and hypnogram
    with open(psg_log, 'r') as f:
        psg_log_lines = f.readlines()
        psg_recordings = [re.match(psg_recording_regex, line).group(1)
                          for line in psg_log_lines
                          if re.match(psg_recording_regex, line)]

    with open(hypno_log, 'r') as f:
        hypno_log_lines = f.readlines()
        hypno_recordings = [re.match(hypno_recording_regex, line).group(1)
                            for line in hypno_log_lines
                            if re.match(hypno_recording_regex, line)]

    shared_recordings = set(psg_recordings).intersection(set(hypno_recordings))
    if len(shared_recordings) == len(psg_recordings) == len(hypno_recordings):
        logging.info('All recordings have both PSG and hypnogram')
    else:
        logging.info(f'PSGs missing hypnogram: {set(psg_recordings) - shared_recordings}')
        logging.info(f'Hypnograms missing PSG: {set(hypno_recordings) - shared_recordings}')

    # check if all recordings have the required channels
    psg_config_file = glob(os.path.join(os.path.split(psg_log)[0], '.hydra', 'config.yaml'))[0]
    with open(psg_config_file, 'r') as f:
        psg_config = yaml.safe_load(f)
        required_eeg_channels = (
            list(psg_config['psg']['eeg_channels'])
            if psg_config['psg']['renamed_eeg_channels'] is False
            else list(psg_config['psg']['renamed_eeg_channels'])
        )
        required_eog_channels = (
            list(psg_config['psg']['eog_channels'])
            if psg_config['psg']['renamed_eog_channels'] is False
            else list(psg_config['psg']['renamed_eog_channels'])
        )
        required_emg_channels = (
            list(psg_config['psg']['emg_channels'])
            if psg_config['psg']['renamed_emg_channels'] is False
            else list(psg_config['psg']['renamed_emg_channels'])
        )
        required_channels = required_eeg_channels + required_eog_channels + required_emg_channels

    psg_log_channel_matches = [re.match(psg_channels_regex, line).groups()
                               for line in psg_log_lines
                               if re.match(psg_channels_regex, line)]
    check2_ok = True
    for recording, (n_channels, channels) in zip(psg_recordings, psg_log_channel_matches):
        parsed_channels = ast.literal_eval(channels)
        if set(parsed_channels) != set(required_channels):
            logging.info(f'{recording} missing channels: {set(required_channels) - set(parsed_channels)}')
            check2_ok = False

    if check2_ok:
        logging.info('All recordings have the required channels')

    logging.info('')